# MVP Soccer Match Prediction Model

This notebook demonstrates loading the `game_features.parquet` dataset, performing exploratory data analysis, and training an MVP Neural Network (MLPClassifier) for predicting match outcomes (W/D/L).


## 1. Imports and Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)


## 2. Load Data


In [ ]:
# Determine the path to the data file
# Try relative path first
data_path = Path("../../Data/game_features.parquet")

# Alternative paths if running from different locations
if not data_path.exists():
    data_path = Path("PROJECT 3/Code/Data/game_features.parquet")
if not data_path.exists():
    data_path = Path("../Data/game_features.parquet")
if not data_path.exists():
    # Absolute path as last resort
    import os
    data_path = Path(os.getcwd()) / "PROJECT 3" / "Code" / "Data" / "game_features.parquet"

print(f"Loading data from: {data_path}")

# Load the Parquet file
df = pd.read_parquet(data_path)

print(f"\nDataset loaded successfully!")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
print(f"\nFirst few rows:")
df.head()


## 3. Exploratory Data Analysis (EDA)


### 3.1 Target Variable Distribution


In [ ]:
# Check target variable distribution
if 'RESULT' in df.columns:
    target_dist = df['RESULT'].value_counts()
    print("Target variable distribution:")
    print(target_dist)
    print(f"\nPercentages:")
    print(target_dist / len(df) * 100)
    
    # Visualize
    plt.figure(figsize=(8, 5))
    target_dist.plot(kind='bar', color=['#1f77b4', '#ff7f0e', '#2ca02c'])
    plt.title('Distribution of Match Results (RESULT)')
    plt.xlabel('Result')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("RESULT column not found in dataset")


### 3.2 Data Types and Missing Values


In [ ]:
# Check data types
print("Data types:")
print(df.dtypes.value_counts())
print(f"\nTotal columns: {len(df.columns)}")

# Check missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print("\nColumns with missing values:")
print(missing_df[missing_df['Missing Count'] > 0].head(20))


### 3.3 Descriptive Statistics


In [ ]:
# Get numeric predictor columns (exclude metadata and target)
metadata_cols = [col for col in df.columns if col.startswith('ID_')]
target_col = 'RESULT' if 'RESULT' in df.columns else None

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
predictor_cols = [col for col in numeric_cols if col not in metadata_cols and col != target_col]

print(f"Metadata columns: {len(metadata_cols)}")
print(f"Target column: {target_col}")
print(f"Numeric predictor features: {len(predictor_cols)}")

# Display descriptive statistics for a sample of predictors
if predictor_cols:
    sample_cols = predictor_cols[:10]  # First 10 predictors
    print(f"\nDescriptive statistics (sample of {len(sample_cols)} predictors):")
    print(df[sample_cols].describe())


## 4. Data Preprocessing


### 4.1 Prepare Features and Target


In [ ]:
# Separate metadata, target, and predictors
metadata_cols = [col for col in df.columns if col.startswith('ID_')]
target_col = 'RESULT' if 'RESULT' in df.columns else None

# Get all numeric predictor columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
predictor_cols = [col for col in numeric_cols if col not in metadata_cols and col != target_col]

# Extract features and target
X = df[predictor_cols].copy()
y = df[target_col].copy() if target_col else None

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape if y is not None else 'N/A'}")

# Filter out predictors with >50% missing values
missing_threshold = 0.5
missing_pct = X.isnull().sum() / len(X)
valid_cols = missing_pct[missing_pct <= missing_threshold].index.tolist()

print(f"\nPredictors with >{missing_threshold*100}% missing: {len(predictor_cols) - len(valid_cols)}")
X = X[valid_cols]
predictor_cols = valid_cols

print(f"Valid predictors after filtering: {len(predictor_cols)}")


### 4.2 Handle Missing Values and Encode Target


In [ ]:
# Fill remaining missing values with median
X_filled = X.fillna(X.median())

print(f"Missing values after filling: {X_filled.isnull().sum().sum()}")

# Encode target variable
if y is not None:
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    
    print(f"\nTarget encoding:")
    print(f"  Classes: {le.classes_}")
    print(f"  Encoded: {np.unique(y_encoded, return_counts=True)}")
else:
    y_encoded = None
    print("No target variable found")


### 4.3 Train-Test Split (Temporal Split)


In [ ]:
# Split into train and test sets (using temporal split if ID_DATE exists)
if 'ID_DATE' in df.columns and target_col:
    # Temporal split: use earlier games for training, later for testing
    dates = pd.to_datetime(df.loc[X.index, 'ID_DATE'])
    split_date = dates.quantile(0.8)  # 80% for training
    
    train_mask = dates < split_date
    test_mask = dates >= split_date
    
    X_train = X_filled[train_mask]
    X_test = X_filled[test_mask]
    y_train = y_encoded[train_mask]
    y_test = y_encoded[test_mask]
    
    print(f"Temporal split:")
    print(f"  Split date: {split_date.date()}")
    print(f"  Training set: {len(X_train):,} games ({len(X_train)/len(X_filled)*100:.1f}%)")
    print(f"  Test set: {len(X_test):,} games ({len(X_test)/len(X_filled)*100:.1f}%)")
else:
    # Random split
    X_train, X_test, y_train, y_test = train_test_split(
        X_filled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
    )
    print(f"Random split:")
    print(f"  Training set: {len(X_train):,} games ({len(X_train)/len(X_filled)*100:.1f}%)")
    print(f"  Test set: {len(X_test):,} games ({len(X_test)/len(X_filled)*100:.1f}%)")


### 4.4 Scale Features


In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Features scaled:")
print(f"  Training shape: {X_train_scaled.shape}")
print(f"  Test shape: {X_test_scaled.shape}")


## 5. MVP Neural Network Model (MLPClassifier)


For this MVP, we'll use a Multi-Layer Perceptron (MLPClassifier) from scikit-learn as a simplified version of a neural network. This is a good starting point before implementing a more complex RNN/LSTM model.


In [ ]:
# Train MLPClassifier
print("Training MLPClassifier...")

mlp = MLPClassifier(
    hidden_layer_sizes=(100, 50),  # Two hidden layers: 100 and 50 neurons
    activation='relu',
    solver='adam',
    alpha=0.0001,  # L2 regularization
    batch_size='auto',
    learning_rate='constant',
    learning_rate_init=0.001,
    max_iter=500,
    shuffle=True,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    verbose=True
)

mlp.fit(X_train_scaled, y_train)
print("\nTraining complete!")


In [ ]:
# Make predictions
y_train_pred = mlp.predict(X_train_scaled)
y_test_pred = mlp.predict(X_test_scaled)

# Calculate accuracies
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("Model Performance:")
print(f"  Training Accuracy: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")
print(f"  Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")


### 5.2 Classification Report


In [ ]:
# Detailed classification report
if y is not None and hasattr(le, 'classes_'):
    class_names = le.classes_
    print("\nClassification Report (Test Set):")
    print(classification_report(y_test, y_test_pred, target_names=class_names))
else:
    print("\nClassification Report (Test Set):")
    print(classification_report(y_test, y_test_pred))


### 5.3 Confusion Matrix


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_test_pred)

if y is not None and hasattr(le, 'classes_'):
    class_names = le.classes_
else:
    class_names = [f"Class {i}" for i in range(len(np.unique(y_test)))]

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix (Test Set)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

print(f"\nConfusion Matrix:")
print(cm)


### 5.4 Model Summary


In [ ]:
print("Model Architecture Summary:")
print(f"  Input features: {X_train_scaled.shape[1]}")
print(f"  Hidden layers: {mlp.hidden_layer_sizes}")
print(f"  Output classes: {mlp.n_outputs_}")
print(f"  Total iterations: {mlp.n_iter_}")
print(f"  Loss: {mlp.loss_:.4f}")


## 6. Summary and Next Steps

### Summary
- Successfully loaded the `game_features.parquet` dataset
- Performed exploratory data analysis
- Preprocessed the data (handled missing values, encoded target, scaled features)
- Trained an MVP MLPClassifier model
- Evaluated the model performance

### Next Steps for Improvement
1. **Feature Engineering**: Explore feature importance, remove redundant features
2. **Hyperparameter Tuning**: Use GridSearchCV or RandomizedSearchCV
3. **True RNN/LSTM**: Implement sequence models to capture temporal dependencies
4. **Ensemble Methods**: Combine multiple models for better performance
5. **Cross-Validation**: Use time-series cross-validation for more robust evaluation
6. **Feature Selection**: Use techniques like mutual information or permutation importance
